# 06. 실습: EXA로 만드는 병렬·직렬·연쇄 리서치 Deep Agent

오늘 배운 흐름, 계약, 예산, 측정을 **공개 웹 리서치**에 옮깁니다. 같은 주제를 같은 소주제로 나눈 뒤 세 구조로 실행하고, 시간이 달라지는 효과와 정보 흐름이 달라지는 효과를 따로 읽습니다. 끝에는 오늘의 설계 기록을 모아 이 리서치에 쓸 구조를 직접 고르고, 전혀 다른 업무로 옮겨 봅니다. 이 실습은 점수를 매기지 않습니다. 실패한 기록도 이 실습의 결과입니다.

## 학습 목표

- 소주제 하나를 맡는 Deep Agent Worker를 조립해 같은 주제로 실행합니다.
- 시간 효과와 정보 흐름 효과를 구분하기 위해 병렬, 직렬(같은 입력), 연쇄(앞 결과 전달) 셋으로 나눠 비교합니다.
- 외부로 나가는 질의와 돌아오는 자료의 경계를 코드로 다룹니다.
- 자동 검증과 사람의 원문 대조를 따로 적습니다.
- 오늘 쌓은 설계 기록을 근거로 구조를 고르고 다른 업무에 옮깁니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 주제와 분해, 내 예측 | 7 |
| 2 | EXA 도구와 외부 경계 | 8 |
| 3 | Worker 한 개 | 7 |
| 4 | 병렬과 직렬 | 13 |
| 5 | 연쇄 | 8 |
| 6 | 자동 검증과 원문 대조 | 8 |
| 7 | 구조 선택과 전이 | 6 |
| 8 | 직접 해보기와 정리 | 3 |

## 호출 규모

- 이 실습은 실제 모델과 실제 EXA만 씁니다. `ANTHROPIC_API_KEY`나 `EXA_API_KEY`가 없으면 첫 코드 셀이 안내와 함께 멈춥니다. 키 없이 대신 돌아가는 길은 없습니다.
- 기본 실행의 상한은 모델 호출 약 53회, EXA 호출 약 21회입니다. Worker가 검색을 한 메시지에 묶으면 모델 호출은 이보다 적게 나옵니다.
- **EXA는 수강생 본인의 크레딧을 씁니다.** 유료 셀은 실행 직전에 `announce`가 규모를 한 줄로 알립니다.
- `직접 해보기`의 유료 셀을 모두 켜면 모델 호출 약 40~64회와 EXA 호출 약 12~24회가 더해집니다. 선택 셀 `RUN_OPTIONAL`을 켜면 모델 호출 약 10~16회와 EXA 호출 약 3~6회가 더해집니다. 둘 다 기본값은 꺼짐입니다.
- 웹 결과는 실행할 때마다 달라질 수 있습니다. 저장해 둔 결과를 재생하지 않습니다. 달라지는 것 자체가 이 실습의 관찰 항목입니다.
- 이 노트북의 모든 수치는 **1회 관찰**입니다. 통계로 읽지 않습니다.

In [ ]:
import inspect
import json
import re
from collections import Counter
from typing import TypedDict

import httpx
from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langgraph.graph import END, START, StateGraph

from mas_lab.contracts import plan_hash
from mas_lab.exa import ExaClient, allowed_url, assert_public_query, make_exa_tool
from mas_lab.research import (WORKER_PROMPT, Claim, Finding, ReportSection, ResearchPlan, ResearchReport, SubTopic,
                              build_research_graph, run_research, url_overlap, validate_report)
from mas_lab.trace import ModelMeter, Trace, overlap_seconds, timeline
from mas_lab.view import bars, flows_of, message_table, read_design_log, show_graph, show_table, write_design_log
from mas_lab.workers import DENY_WRITES, allowlist_middleware
from rag_lab.evidence import Budget
from rag_lab.settings import announce, key_status, make_model, require_exa_key, require_key

require_key()
require_exa_key()
model = make_model()


def shown(value, digits=None):
    # 없는 값을 0으로 읽지 않게 표시합니다. 비용과 토큰 칸은 모두 이 함수를 거칩니다.
    if value is None:
        return "확인 안 됨"
    return round(value, digits) if digits is not None else value


show_table([key_status()])

## 1. 주제와 분해, 내 예측

**문제.** 리서치 시스템을 Sequential과 Parallel 두 가지로 만들어 비교하면 답하기 어려운 질문이 남습니다. 두 결과가 다를 때, 그 차이가 **동시에 돌아서** 생긴 것일까요, **앞 Worker의 결과를 뒤 Worker가 읽어서** 생긴 것일까요? 흔히 Sequential이라고 부르는 구조에는 이 둘이 섞여 있습니다. 순서대로 돌리면서 앞 결과를 넘기기 때문입니다.

그래서 이 실습은 Sequential을 둘로 나눕니다.

| 구조 | 분해 | 뒤 Worker의 입력 | Worker끼리 동시 실행 | 앞 구조와 비교해 바꾸는 것 |
|---|---|---|---|---|
| `parallel` | 같음 | 소주제만 | 예 | 기준 |
| `serial` | 같음 | 소주제만 | 아니요 | `parallel`과 비교하면 **동시 실행 여부**를 바꿉니다. 시간 효과를 보는 짝입니다 |
| `chained` | 같음 | 소주제 + 앞 Worker의 결과 | 아니요 | `serial`과 비교하면 **앞 결과의 전달**을 바꿉니다. 정보 흐름 효과를 보는 짝입니다 |

`parallel`과 `serial`은 **같은 그래프**입니다. 그래프를 고치지 않고 `config={"max_concurrency": 1}`만 다르게 줍니다. "분해가 같다"를 말이 아니라 코드와 `plan_hash`로 보장하기 위해서입니다.

주의할 점도 있습니다. 구조 말고 달라지는 것이 둘 더 있습니다. 모델은 같은 입력에도 다른 검색어를 고를 수 있고, 웹은 실행마다 다른 문서를 돌려줄 수 있습니다. 그러니 두 구조의 차이를 곧바로 구조 탓으로 돌릴 수 없습니다. 이 점은 4절에서 URL 겹침으로 직접 재 봅니다.

주제는 오늘 다룬 질문, 곧 멀티에이전트 설계 패턴을 어떤 기준으로 고를 것인가입니다. 공개 문서와 논문으로 답할 수 있는 질문이고, 고객 이름이나 내부 번호가 들어갈 자리가 없습니다.

In [ ]:
TOPIC = "멀티에이전트 설계 패턴을 어떤 기준으로 고를 것인가"
PLAN = ResearchPlan(topic=TOPIC, subtopics=[
    SubTopic(id="s1", question="Supervisor와 Orchestrator-Worker 구조는 어떤 상황에 맞고 어떤 한계가 있는가"),
    SubTopic(id="s2", question="Handoff 구조는 어떤 상황에 맞고, 담당자가 바뀔 때 Context는 어떻게 전달되고 어디서 손실되는가"),
    SubTopic(id="s3", question="단일 Agent로 충분한 경우와 멀티에이전트가 비용만 늘리는 경우를 공개 문서와 논문은 어떻게 구분하는가"),
])
MAX_SUBTOPICS = 3
MAX_SEARCHES_PER_WORKER = 2
NUM_RESULTS = 3
MAX_CHARS = 2500
ALLOWED_DOMAINS = frozenset({"docs.langchain.com", "langchain-ai.github.io", "anthropic.com", "arxiv.org"})
RUN_OPTIONAL = False

assert len(PLAN.subtopics) <= MAX_SUBTOPICS, "소주제를 늘리면 호출 상한도 함께 늘어납니다."
assert_public_query(TOPIC)
RUN_ARGS = {"domains": ALLOWED_DOMAINS, "per_worker": MAX_SEARCHES_PER_WORKER, "num_results": NUM_RESULTS, "max_chars": MAX_CHARS}
workers = len(PLAN.subtopics)
show_table([{"plan_hash": plan_hash(PLAN), "소주제": workers, "실행당 EXA 상한": workers * MAX_SEARCHES_PER_WORKER,
             "실행당 모델 예상 상한": workers * (MAX_SEARCHES_PER_WORKER + 1) + 1, "허용 도메인": len(ALLOWED_DOMAINS)}])
print("모델 호출 0회, EXA 호출 0회")

**먼저 예측하세요.** 실행하기 전에 아래 표를 채웁니다. 결과를 본 뒤에 이유를 지어내지 않기 위해서입니다. 숫자가 아니라 "가장 빠름", "비슷함", "가장 많음"처럼 적어도 됩니다.

| 항목 | `parallel` | `serial` | `chained` |
|---|---|---|---|
| 전체 시간 | | | |
| 입력 토큰 | | | |
| 다른 구조와의 URL 겹침 | | | |
| 보고서 품질 (무엇을 보고 판단할지까지) | | | |

그리고 한 문장을 더 적습니다. **위 세 소주제는 서로 독립적인가요?** s3의 답이 s1이나 s2의 답에 기대는 부분이 있다고 보나요? 독립적이라고 답했다면 `chained`에서 무엇을 얻을 수 있다고 예상하나요?

## 2. EXA 도구와 외부 경계

**문제.** 사내 업무를 다루던 Agent에게 웹 검색을 붙이면 경계가 하나 새로 생깁니다. 검색어는 회사 밖으로 나가고, 검색 결과는 밖에서 안으로 들어옵니다. 어느 쪽도 모델의 선의에 맡길 수 없습니다.

| 방향 | 지나가는 것 | 위험 | 이 노트북의 장치 |
|---|---|---|---|
| 밖으로 | 모델이 정한 검색어 | 고객 이름, 결제 번호, 내부 문서 ID가 함께 나감 | `assert_public_query`가 호출 전에 거절 |
| 안으로 | 웹 문서 본문 | 믿을 수 없는 곳의 글, 문서 안의 지시문 | `allowed_url`, `source_id`, Worker 프롬프트 |
| 안으로 | 본문의 길이 | Context 비용이 늘어남 | `MAX_CHARS`, `NUM_RESULTS` |

**내부 질문 전체를 보내지 않습니다.** 고객 요청을 그대로 검색어로 넣는 것이 가장 쉬운 구현이지만, 그 순간 고객 정보가 외부 검색 서비스로 나갑니다. 외부에 물을 것은 "우리 고객의 이 결제"가 아니라 "이런 종류의 일을 일반적으로 어떻게 다루는가"입니다.

아래 셀은 호출 없이 이 경계를 먼저 실패시킵니다. 마지막 질의는 일부러 고객 이름과 번호 없이 쓴 문장입니다. 코드가 뜻까지 판단하지 못한다는 것을 보기 위해서입니다.

In [ ]:
# 실패 재현: 내부 정보가 든 질의와 위장 도메인을 호출 전에 거절합니다.
query_rows = []
for query in ["알파 고객 P-9002 중복 결제 환불", "INV-2026-08 청구서 내역", "지난달 두 번 결제한 우리 고객의 환불 기한"]:
    try:
        assert_public_query(query)
        verdict = "통과: 밖으로 나갑니다"
    except ValueError as error:
        verdict = f"거절: {error}"
    query_rows.append({"질의": query, "판정": verdict})
show_table(query_rows)

urls = ["https://docs.langchain.com/oss/python/langgraph", "https://docs.langchain.com.evil.test/x",
        "https://evil.test/docs.langchain.com", "https://user:pw@docs.langchain.com/", "file:///etc/passwd"]
show_table([{"URL": url, "허용": allowed_url(url, ALLOWED_DOMAINS)} for url in urls])
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
announce("EXA 검색 1회로 자료 형태 확인", exa_calls=1)
probe = ExaClient(trace=Trace("probe"), max_searches=1, domains=ALLOWED_DOMAINS, num_results=NUM_RESULTS, max_chars=MAX_CHARS)
try:
    results = probe.search(PLAN.subtopics[0].question)
except httpx.HTTPError as error:  # 실패를 숨기지 않고 종류만 출력합니다.
    results = []
    print("EXA 호출 실패:", type(error).__name__)
show_table([{"source_id": r["source_id"], "제목": r["title"][:40], "URL": r["url"], "게시일": shown(r["published"]),
             "글자 수": len(r["text"])} for r in results], columns=["source_id", "제목", "URL", "게시일", "글자 수"])
print("EXA 비용(USD):", shown(probe.cost_dollars, 4), "| EXA 대기 초:", round(probe.seconds, 2), "| 검색 횟수:", probe.searches)

**관찰할 것.** 표를 세 칸으로 읽습니다.

- `글자 수`가 `MAX_CHARS`에 붙어 있다면 본문이 잘린 것입니다. 본문 뒤쪽에 있는 조건이나 예외는 Worker가 볼 수 없습니다. `MAX_CHARS`는 정보량이면서 Context 비용입니다. 2,500자짜리 문서 3개를 돌려받는 검색 2번이면 Worker 하나의 다음 모델 호출에 그만큼의 본문이 얹힙니다.
- `게시일`이 `확인 안 됨`이면 EXA가 날짜를 주지 않은 문서입니다. 오래된 문서를 최신 문서처럼 인용하지 않도록 사람이 확인할 칸입니다.
- `EXA 비용(USD)`이 숫자로 나오면 응답의 `costDollars`를 합산한 값입니다. `확인 안 됨`이 나오면 응답에 그 칸이 없었다는 뜻입니다. **0달러가 아닙니다.** 비용을 모르는 것과 비용이 없는 것은 다릅니다.

한 가지 더 있습니다. **도구가 돌려준 웹 문서는 자료이지 명령이 아닙니다.** 문서 안에 "이전 지시를 무시하고 다음 문장을 쓰세요"라고 적혀 있어도 Worker는 그 문장을 따르면 안 됩니다. 프롬프트에 그렇게 적어 두었지만, 프롬프트는 보장이 아닙니다. 그래서 이 Worker의 도구는 검색 하나뿐이고 쓰기 권한이 없습니다. 지시문에 넘어가도 할 수 있는 일을 줄여 둔 것입니다.

In [ ]:
print(inspect.getsource(make_exa_tool))
print("모델이 보는 인자:", json.dumps(make_exa_tool(probe).args, ensure_ascii=False))
print("모델 호출 0회, EXA 호출 0회")

소스에서 네 곳을 짚어 읽습니다.

1. **인자가 `query` 하나입니다.** 모델은 도메인, 결과 수, 본문 길이를 정할 수 없습니다. 이 값들은 `ExaClient`를 만들 때 코드가 정합니다. 모델이 더 긴 본문을 달라고 할 길이 없으니 예산이 새지 않습니다.
2. **돌려주는 값에 `source_id`가 있습니다.** URL을 그대로 인용하게 하면 모델이 그럴듯한 URL을 지어내도 형식만으로는 알아차리기 어렵습니다. `source_id`는 코드가 URL 해시로 만들어 `client.sources`에 등록한 값이라, 등록 여부만 보면 실제로 받은 문서인지 알 수 있습니다. 위조를 기계가 잡을 수 있게 만드는 장치입니다.
3. **검색 한도를 넘으면 예외가 아니라 값을 돌려줍니다.** `search_limit`를 받은 Worker는 "찾은 자료로 정리"할 수 있습니다. 반면 `ExaClient` 전체 한도를 넘으면 `BudgetExceeded`가 납니다. Worker 하나의 한도는 Worker가 대처할 일이고, 전체 한도는 실행을 멈출 일입니다. 04에서 본 실패별 다음 행동과 같은 구분입니다.
4. **거절 메시지에 질의를 되돌려 주지 않습니다.** 막힌 질의에 들어 있던 고객 이름이 메시지를 타고 기록에 남지 않게 하기 위해서입니다.

## 3. Worker 한 개

**문제.** 병렬로 늘리기 전에 Worker 하나가 약속을 지키는지부터 봅니다. 하나가 근거 없는 주장을 쓴다면 셋으로 늘려도 근거 없는 주장이 세 배가 될 뿐입니다.

Worker의 계약은 두 스키마입니다.

| 스키마 | 칸 | 뜻 |
|---|---|---|
| `Finding` | `subtopic_id` | 어느 소주제의 결과인지. 코드가 덮어씁니다 |
| `Finding` | `claims` | 주장의 목록 |
| `Finding` | `uncertainty` | 찾지 못한 것. 비어 있는 것이 언제나 좋은 신호는 아닙니다 |
| `Claim` | `text` | 주장 한 문장 |
| `Claim` | `source_ids` | 이 주장의 근거가 된 `source_id` |

Deep Agent를 쓰지만 기본 도구는 모두 숨기고 `web_search`만 보이게 합니다. `allowlist_middleware`가 모델 층에서 도구를 가리고, `permissions=DENY_WRITES`가 쓰기를 막습니다. 이 Worker가 아래 셀에서 어떻게 조립되는지 그대로 읽어 보세요. 그래프 안의 Worker와 같은 부품입니다.

**먼저 예측하세요.** 검색 한도는 2회입니다. 이 Worker는 검색을 몇 번 할까요? 한 번에 하나씩 할까요, 한 메시지에 묶어서 할까요? `uncertainty`는 비어 있을까요? 표를 보기 전에 적습니다.

In [ ]:
announce("Worker 1개로 소주제 하나 조사", model_calls=5, exa_calls=2)
single_client = ExaClient(trace=Trace("single"), max_searches=MAX_SEARCHES_PER_WORKER, domains=ALLOWED_DOMAINS,
                          num_results=NUM_RESULTS, max_chars=MAX_CHARS)
single_agent = create_deep_agent(
    model=model, tools=[make_exa_tool(single_client, MAX_SEARCHES_PER_WORKER)], system_prompt=WORKER_PROMPT,
    middleware=allowlist_middleware(Budget(max_model_calls=8, max_tool_calls=4, max_seconds=180), frozenset({"web_search"})),
    permissions=DENY_WRITES, response_format=ProviderStrategy(Finding), name="research-one")
single_meter, target = ModelMeter(), PLAN.subtopics[0]
try:
    single_state = single_agent.invoke({"messages": [{"role": "user", "content": f"소주제: {target.question}"}]},
                                       config={"callbacks": [single_meter], "recursion_limit": 30})
except Exception as error:  # 429도 실패로 드러내고 뒤 셀은 계속 읽습니다.
    print("Worker 실패:", type(error).__name__)
    single_state = {"messages": []}
single_finding = (single_state.get("structured_response") or Finding(uncertainty=["구조화 응답 없음"])).model_copy(update={"subtopic_id": target.id})
show_table(message_table(single_state["messages"]))
totals = single_meter.totals()
print("모델 호출:", totals["model_calls"], "| 입력 토큰:", shown(totals["input_tokens"]), "| 검색어:", single_client.queries)
print("EXA 호출:", single_client.searches, "| EXA 비용(USD):", shown(single_client.cost_dollars, 4))
show_table([{"주장": c.text[:70], "source_ids": ", ".join(c.source_ids)} for c in single_finding.claims], columns=["주장", "source_ids"])
print("uncertainty:", single_finding.uncertainty)

In [ ]:
# 실패 재현: Finding의 source_ids가 실제로 받은 출처인지 확인합니다. 위조 source_id를 넣은 사본은 거절됩니다.
def finding_problems(finding, sources):
    wrapped = ResearchReport(summary="", sections=[ReportSection(heading=finding.subtopic_id, claims=finding.claims)])
    return validate_report(wrapped, sources)


forged = single_finding.model_copy(deep=True)
forged.claims.append(Claim(text="문서에 없는 주장을 그럴듯한 번호로 인용합니다", source_ids=["S-00000000"]))
forged.claims.append(Claim(text="출처를 붙이지 않은 주장입니다", source_ids=[]))
show_table([{"대상": "Worker 원본", "문제 수": len(finding_problems(single_finding, single_client.sources))},
            {"대상": "위조 사본", "문제 수": len(finding_problems(forged, single_client.sources))}])
print("\n".join(finding_problems(forged, single_client.sources)))
print("등록된 출처 수:", len(single_client.sources), "| 모델 호출 0회, EXA 호출 0회")

**관찰할 것.** 앞 셀 `message_table`의 `인자` 칸에서 **검색어를 누가 정했는지** 읽습니다. 소주제 문장을 그대로 썼다면 질문이 곧 검색어가 된 것이고, 영어 용어로 바꿨다면 모델이 문서 쪽 언어에 맞춘 것입니다. 어느 쪽이든 코드가 아니라 모델의 판단입니다.

**몇 번 어떻게 검색했는지**도 읽습니다. `AI` 행 하나의 `도구` 칸에 `web_search`가 두 번 있으면 한 메시지에서 검색을 묶은 것이고, `AI` 행과 `Tool` 행이 번갈아 나오면 앞 결과를 보고 다음 검색어를 정한 것입니다. 묶으면 모델 호출이 줄지만 앞 결과로 검색어를 고칠 기회는 없습니다. 이 선택도 모델이 합니다.

**`uncertainty`에 무엇을 적었는지** 읽습니다. 구체적으로 "무엇을 찾지 못했다"고 적었다면 다음 조사의 출발점이 됩니다. 비어 있다면 다 찾았다는 뜻일 수도, 빈 곳을 알아차리지 못했다는 뜻일 수도 있습니다. 비어 있음을 품질의 증거로 읽지 않습니다.

마지막으로 `문제 수` 표를 읽습니다. 원본이 0이고 위조 사본이 2라면 검사가 형식 위조를 잡은 것입니다. 원본이 0이 아니라면 실제 모델이 등록되지 않은 값을 쓰거나 출처 없는 주장을 냈다는 뜻입니다. 지우지 말고 어떤 주장인지 읽습니다. 다만 **이 검사는 번호가 등록되었는지만 봅니다.** 그 문서가 그 주장을 정말 뒷받침하는지는 6절에서 사람이 봅니다.

## 4. 병렬과 직렬

**문제.** 병렬이 얼마나 빠른지 재려면 다른 조건은 같아야 합니다. 그래서 그래프를 두 벌 만들지 않습니다. 아래 소스에서 세 부분을 찾아 읽습니다.

- **`Send`.** `START`에서 소주제마다 `Send("worker", ...)`를 하나씩 보냅니다. 03에서 본 병렬 fan-out입니다.
- **reducer.** `_State`의 `findings`가 `merge_results`로 합쳐집니다. 동시에 돌아온 결과를 소주제 ID로 모으고, 같은 ID의 상충 결과는 거절합니다. 도착 순서가 결과를 바꾸지 않습니다.
- **`synth`.** 합성 노드는 모델을 한 번 부릅니다. 프롬프트가 "조사 결과에 있는 주장과 source_ids만 쓰세요"라고 요구합니다. 요구일 뿐이므로 6절에서 다시 검사합니다.

직렬은 `run_research`가 `config["max_concurrency"] = 1`을 주는 것이 전부입니다.

**시간은 세 칸으로 나눠 봅니다.** `전체 초`는 벽시계 시간이고, `EXA 초 합`과 `모델 초 합`은 각 호출이 걸린 시간을 더한 값입니다. 병렬에서는 동시에 도는 구간이 두 번 더해지므로 **합이 전체보다 클 수 있습니다.** 오류가 아니라 병렬의 정의입니다. 합이 전체보다 작다면 측정하지 않은 시간이 있다는 뜻입니다. 그래프와 Agent 내부의 처리 시간이 여기에 들어갑니다.

**먼저 예측하세요.** Worker가 3개이니 병렬이 직렬보다 3배 빠를까요? 합성은 어느 구조에서든 한 번 혼자 돕니다. 이 구간이 몫을 얼마나 차지한다고 보나요? 그리고 전체 시간 가운데 EXA를 기다리는 몫은 얼마일까요? 적어 두고 실행합니다.

In [ ]:
print(inspect.getsource(build_research_graph))
show_graph(build_research_graph(PLAN, ExaClient(max_searches=0), model))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
announce("병렬 리서치", model_calls=16, exa_calls=6)
parallel = run_research("parallel", PLAN, model=model, **RUN_ARGS)
print("상태:", parallel["status"], "| 오류:", parallel["errors"])
print("전체 초:", parallel["elapsed_seconds"], "| 모델 호출:", parallel["model_calls"], "| EXA 호출:", parallel["exa_calls"])

In [ ]:
def read_report(result):
    # 실패한 실행도 같은 칸으로 읽습니다. 보고서가 없으면 없다고 출력합니다.
    print(timeline([s for s in result["spans"] if s["kind"] in {"worker", "exa"}]))
    report = result["report"]
    print("\n보고서 요약:", report["summary"][:280] if report else "보고서가 없습니다.")
    rows = [{"소주제": sub.id, "주장 수": len(found.get("claims", [])), "불확실성 수": len(found.get("uncertainty", [])),
             "인용한 출처 수": len({sid for claim in found.get("claims", []) for sid in claim["source_ids"]})}
            for sub in PLAN.subtopics for found in [result["findings"].get(sub.id, {})]]
    show_table(rows)
    print("등록된 출처 수:", len(result["sources"]), "| 검색어 수:", len(result["queries"]))


read_report(parallel)
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
announce("같은 그래프를 max_concurrency=1로 실행", model_calls=16, exa_calls=6)
serial = run_research("serial", PLAN, model=model, **RUN_ARGS)
print("상태:", serial["status"], "| 오류:", serial["errors"])
print("전체 초:", serial["elapsed_seconds"], "| 모델 호출:", serial["model_calls"], "| EXA 호출:", serial["exa_calls"])
print(timeline([s for s in serial["spans"] if s["kind"] in {"worker", "exa"}]))

In [ ]:
def after_workers(result):
    # 마지막 Worker가 끝난 뒤의 시간입니다. 대부분 혼자 도는 합성 노드입니다.
    ends = [s["end"] for s in result["spans"] if s["kind"] == "worker"]
    return round(result["elapsed_seconds"] - max(ends), 3) if ends else None


def compare_rows(results):
    return [{"구조": r["mode"], "상태": r["status"], "plan_hash": r["plan_hash"], "전체 초": r["elapsed_seconds"], "EXA 초 합": r["exa_seconds"],
             "모델 초 합": r["model_seconds"], "마지막 Worker 이후 초": shown(after_workers(r)), "Worker 겹침 초": r["worker_overlap_seconds"],
             "EXA 겹침 초": overlap_seconds([s for s in r["spans"] if s["kind"] == "exa"]), "입력 토큰": shown(r["input_tokens"]),
             "출력 토큰": shown(r["output_tokens"]), "usage 누락": r["usage_missing"], "모델 호출": r["model_calls"], "EXA 호출": r["exa_calls"],
             "EXA 비용(USD)": shown(r["cost_dollars"], 4)} for r in results]


show_table(compare_rows([parallel, serial]))
print("plan_hash 동일:", parallel["plan_hash"] == serial["plan_hash"])
print(bars({"parallel 전체 초": parallel["elapsed_seconds"], "serial 전체 초": serial["elapsed_seconds"]}))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
pair = url_overlap(parallel, serial)
show_table([{"비교": "parallel 대 serial", **pair}])
shared = {s["url"] for s in parallel["sources"].values()} & {s["url"] for s in serial["sources"].values()}
print("같은 검색어:", sorted(set(parallel["queries"]) & set(serial["queries"])) or "없음")
print("겹친 URL:", sorted(shared) or "없음")
print("모델 호출 0회, EXA 호출 0회")

**관찰할 것.** 비교표는 다음 순서로 읽습니다.

1. **`상태`와 `plan_hash`.** `상태`가 `ok`가 아닌 행은 실패 행입니다. 지우지 않고 다른 칸을 그 사실과 함께 읽습니다. `plan_hash`가 같으면 분해가 같은 것입니다. 입력이 같은지가 아니라 **분해가** 같은지를 증명합니다.
2. **`Worker 겹침 초`.** 병렬이 실제로 일어났는지 보는 칸입니다. `parallel`이 0보다 크고 `serial`이 0이면 스위치가 뜻대로 작동한 것입니다. `parallel`인데 0이라면 이름만 병렬입니다.
3. **`전체 초`.** `serial`을 `parallel`로 나눠 봅니다. 그 배수가 Worker 수인 3보다 작다면 함께 돌 수 없는 구간이 있다는 뜻입니다. 가장 오래 걸린 Worker와 혼자 도는 합성이 하한을 만듭니다. `마지막 Worker 이후 초`가 그 합성의 몫입니다. 이 칸이 `전체 초`에서 차지하는 비율이 클수록 Worker를 아무리 동시에 돌려도 줄일 수 있는 시간이 적습니다.
4. **`EXA 초 합`과 `모델 초 합`.** 합이 `전체 초`보다 크면 겹쳐 돌았다는 뜻입니다. 두 칸을 비교하면 어디를 기다리는 시간이 많은지 보입니다. `EXA 초 합`이 `모델 초 합`보다 훨씬 작다면 검색을 줄여도 시간은 크게 줄지 않습니다.
5. **`EXA 겹침 초`.** Worker 안의 검색이 겹쳤는지 봅니다. `serial`에서 이 칸이 0이면 `max_concurrency=1`이 Worker 안의 도구 호출까지 내려간 것이고, 0보다 크면 Worker는 줄을 섰지만 Worker 안의 검색은 동시에 돌았다는 뜻입니다.
6. **`입력 토큰`, `모델 호출`.** 두 구조의 차이는 구조 효과일까요? 입력이 같으므로 큰 구조 차이를 기대할 이유는 없습니다. 차이가 크다면 모델이 고른 검색 방식이나 돌아온 문서 길이를 먼저 의심합니다.

그다음 `url_overlap`을 읽습니다. **그래프와 분해는 같아도 모델과 웹은 달라질 수 있습니다.** `Jaccard`가 낮다면 두 실행은 다른 자료를 읽고 쓴 것이므로 **품질 차이를 구조 탓으로 읽을 수 없습니다.** `Jaccard`가 높다면 자료 차이라는 설명이 약해집니다. 그래도 한 번의 관찰이니 구조 효과라고 단정하지 않습니다. 시간 차이는 겹침 칸이 함께 설명해 주지만, 품질 차이에는 그런 칸이 없습니다.

## 5. 연쇄

**문제.** 소주제가 서로 기대고 있다면 어떨까요? 앞 조사에서 찾은 것을 알면 뒤 조사가 중복을 피하고 빈 곳을 메울 수 있습니다. 대신 값을 치릅니다. 앞 결과가 뒤 입력에 쌓이고, 동시에 돌릴 수 없으며, 앞 Worker의 오류가 뒤로 옮겨 갈 수 있습니다.

`chained`는 앞 Worker의 `Finding`을 JSON으로 뒤 Worker의 입력에 그대로 붙입니다. 둘 다 한 번에 한 Worker씩 돌기 때문에 `serial` 대 `chained`가 **정보 흐름의 효과**를 보는 짝입니다. 다만 완전히 같지는 않습니다. `chained`는 `max_concurrency`를 주지 않으므로, Worker끼리는 줄을 서지만 Worker 안의 한 메시지에 묶인 검색은 겹칠 수 있습니다. 세 구조 표의 `EXA 겹침 초`와 `EXA 초 합`으로 이 차이가 시간에 얼마나 들어오는지 함께 확인합니다.

**먼저 예측하세요.** 다음을 적습니다.

- 셋째 Worker의 입력 바이트는 첫째 Worker의 몇 배쯤 될까요?
- 무엇이 나아지고 무엇이 나빠질까요? (중복 URL, 주장 수, 불확실성, 토큰, 시간)
- 첫째 Worker가 틀린 주장을 썼다면 둘째 Worker는 그것을 고칠까요, 믿고 쌓을까요? 어떤 칸이나 문장에서 그 흔적을 찾을 수 있을까요?

In [ ]:
def later_worker_input(question, prior):
    # 연쇄에서 뒤 Worker가 받는 입력의 형태입니다. 그래프 안의 문장을 옮겨 적었으므로, 연쇄 실행 뒤에 실측 바이트와 맞춰 봅니다.
    return f"소주제: {question}\n앞 Worker의 조사 결과(중복을 피하고 이어서 조사하세요): {json.dumps(prior, ensure_ascii=False)}"


graph_source = inspect.getsource(build_research_graph)
print(graph_source[graph_source.index("    if chained:"):graph_source.index("\n    else:")])
show_graph(build_research_graph(PLAN, ExaClient(max_searches=0), model, chained=True))
sample_text = later_worker_input(PLAN.subtopics[1].question, {"s1": parallel["findings"].get("s1", {})})
print("둘째 Worker가 받을 입력의 예 (병렬의 s1 결과로 미리 만들었습니다):", len(sample_text.encode("utf-8")), "바이트")
print(sample_text[:500])
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
# Worker별 토큰을 읽으려고 모델에 측정기를 하나 더 답니다. 모델 응답은 바뀌지 않습니다.
chain_calls = ModelMeter()
chain_model = model.model_copy(update={"callbacks": [*model.callbacks, chain_calls]})
announce("앞 결과를 뒤 Worker에 넘기는 연쇄", model_calls=16, exa_calls=6)
chained = run_research("chained", PLAN, model=chain_model, **RUN_ARGS)
print("상태:", chained["status"], "| 오류:", chained["errors"])
print("전체 초:", chained["elapsed_seconds"], "| 모델 호출:", chained["model_calls"], "| EXA 호출:", chained["exa_calls"])
read_report(chained)

In [ ]:
def token_sum(calls, key):
    values = [call[key] for call in calls]
    return None if not values or None in values else sum(values)


# 한 Worker 안에서는 메시지가 늘어납니다. 메시지 수가 줄어들면 다음 Worker나 합성이 시작된 것입니다.
segments, last = [], None
for call in chain_calls.calls:
    if last is None or call["messages"] <= last:
        segments.append([])
    segments[-1].append(call)
    last = call["messages"]
names = [sub.id for sub in PLAN.subtopics] + ["합성"]
rows = []
for i, calls in enumerate(segments):
    name = names[i] if i < len(names) else f"추가 {i}"
    rows.append({"순번": i + 1, "담당": name, "연쇄 입력 바이트": chained["worker_input_bytes"].get(name, "해당 없음"),
                 "병렬 입력 바이트": parallel["worker_input_bytes"].get(name, "해당 없음"), "모델 호출": len(calls),
                 "첫 호출 입력 토큰": shown(calls[0]["input_tokens"]), "입력 토큰 합": shown(token_sum(calls, "input_tokens"))})
show_table(rows)
if "s1" in chained["findings"]:
    rebuilt = later_worker_input(PLAN.subtopics[1].question, {"s1": chained["findings"]["s1"]})
    print("s2 입력 바이트 실측:", shown(chained["worker_input_bytes"].get("s2")), "| later_worker_input으로 다시 잰 값:", len(rebuilt.encode("utf-8")))
print(bars({row["담당"]: row["입력 토큰 합"] for row in rows if isinstance(row["입력 토큰 합"], int)}))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
runs = [parallel, serial, chained]
show_table(compare_rows(runs))
# 응답별 결과 수는 기록되지 않으므로 재조회는 '검색 수 x NUM_RESULTS - 고유 URL'로 잡은 상한입니다. 정확한 값이 아닙니다.
show_table([{"구조": r["mode"], "고유 URL": len({s["url"] for s in r["sources"].values()}),
             "재조회 상한": max(0, r["exa_calls"] * NUM_RESULTS - len({s["url"] for s in r["sources"].values()})),
             "같은 검색어 반복": len(r["queries"]) - len(set(r["queries"]))} for r in runs])
show_table([{"비교": f"{a['mode']} 대 {b['mode']}", **url_overlap(a, b)} for a, b in [(parallel, serial), (serial, chained), (parallel, chained)]])
print("검색어(연쇄):", chained["queries"])
print("모델 호출 0회, EXA 호출 0회")

**관찰할 것.** 먼저 `연쇄 입력 바이트`를 순번대로 읽습니다. `병렬 입력 바이트`는 소주제 문장뿐이어서 거의 일정합니다. 연쇄 칸이 뒤로 갈수록 커진다면 앞 결과가 쌓이는 것입니다. `입력 토큰 합`이 그만큼 함께 커지는지 확인합니다. 두 칸의 단위가 다르고, 토큰에는 프롬프트와 도구 스키마와 웹 본문이 함께 들어갑니다. 그러니 바이트가 늘어난 양보다 토큰이 더 크게 움직이는 일도 자연스럽습니다. 특히 `첫 호출 입력 토큰`은 아직 웹 본문이 얹히기 전의 값이라, 순번별 값의 차이가 앞 결과의 몫을 가장 깨끗하게 보여 줍니다. `입력 토큰 합`은 그 Worker의 `모델 호출` 수에도 좌우됩니다. 검색을 나눠 한 Worker는 웹 본문을 실은 호출이 한 번 더 있으니, `모델 호출`이 다른 행끼리 합을 바로 비교하지 않습니다.

**세 구조의 표는 짝으로 읽습니다.** `serial` 대 `parallel`은 시간, `chained` 대 `serial`은 정보 흐름입니다. `chained`와 `parallel`을 바로 비교하면 두 효과가 섞입니다.

**순서가 답을 바꾸는가?** `연쇄`의 검색어를 읽습니다. 뒤 Worker의 검색어가 앞 결과의 용어를 쓰거나 "앞에서 찾지 못한 것"을 겨냥했다면 정보 흐름이 실제로 일한 것입니다. 검색어가 `parallel`과 비슷하다면 앞 결과는 비용만 늘렸을 수 있습니다. `재조회 상한`과 `같은 검색어 반복`이 `serial`보다 작다면 "중복을 피하세요"가 어느 정도 통했다는 신호이고, 비슷하거나 크다면 그렇지 않습니다. 다만 `재조회 상한`은 실제 횟수가 아니라 상한입니다.

**앞 Worker의 오류가 뒤로 번지는지** 보고서에서 찾습니다. 첫 Worker의 주장 중 의심스러운 것을 하나 고르고, 뒤 소주제의 주장에 그 표현이 근거 없이 다시 나오는지 찾습니다. 뒤 Worker가 앞 주장에 자기 검색 결과가 아닌 `source_id`를 그대로 달았다면 인용 ID는 유효한데 그 Worker가 직접 확인한 것은 아닙니다. 찾지 못했다면 "번지지 않았다"가 아니라 "이번 실행에서는 찾지 못했다"고 적습니다.

## 6. 자동 검증과 원문 대조

**문제.** 보고서가 그럴듯하고 인용이 달려 있으면 맞는 것처럼 보입니다. 코드가 잡을 수 있는 것과 사람이 봐야 하는 것을 나누지 않으면, 자동 검사를 통과했다는 사실이 품질의 증거로 둔갑합니다.

| 질문 | 누가 답하는가 | 어디서 |
|---|---|---|
| 주장에 출처가 붙어 있는가 | 코드 | `validate_report` |
| 그 `source_id`가 실제로 받은 문서인가 | 코드 | `validate_report` |
| 같은 URL을 다시 받느라 돈을 썼는가 | 코드 | `재조회 상한`(상한값) |
| 그 문서가 그 주장을 뒷받침하는가 | **사람** | 원문 대조 |
| 주장이 문서의 조건과 예외를 빼먹지 않았는가 | **사람** | 원문 대조 |
| 빠진 관점이 있는가 | **사람** | 원문 대조 |

아래 첫 셀은 출처를 지운 사본으로 검사가 실제로 실패하는 것을 함께 봅니다. 통과만 보면 검사가 작동하는지 알 수 없기 때문입니다. 둘째 셀은 주장을 고릅니다. 무작위로 고르지 않고 **첫 번째 주장과 숫자가 있는 주장 위주로** 고릅니다. 숫자는 틀렸을 때 가장 비싸고, 대조하기도 쉽습니다.

In [ ]:
# 실패 재현: 출처를 지운 보고서 사본이 자동 검증에서 거절되는 것을 세 보고서와 함께 봅니다.
def check_row(label, report, sources, result=None):
    problems = validate_report(report, sources)
    return {"보고서": label, "문제 수": len(problems), "출처 없는 주장": sum(p.startswith("출처 없는") for p in problems),
            "등록되지 않은 source_id": sum(p.startswith("등록되지") for p in problems),
            "재조회 상한": "해당 없음" if result is None else max(0, result["exa_calls"] * NUM_RESULTS - len(sources))}


rows = [check_row(r["mode"], r["report"], r["sources"], r) for r in runs]
base = next((r for r in runs if r["report"]), None)
stripped = json.loads(json.dumps(base["report"])) if base else None
for section in (stripped or {}).get("sections", []):
    for claim in section["claims"]:
        claim["source_ids"] = []
rows.append(check_row("출처를 지운 사본", stripped, base["sources"] if base else {}))
show_table(rows)
for r in runs:
    print(r["mode"], validate_report(r["report"], r["sources"])[:3] or "문제 없음")
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
def excerpt(text, claim, width=320):
    numbers = sorted(re.findall(r"\d[\d,.]*\d|\d", claim), key=len, reverse=True)  # 긴 숫자일수록 자리를 잘 짚습니다.
    at = next((text.find(n) for n in numbers if n in text), 0)
    return text[max(0, at - 80):max(0, at - 80) + width].replace("\n", " ")


REVIEW_ITEMS = []
for r in runs:
    claims = [c for s in (r["report"] or {}).get("sections", []) for c in s["claims"]]
    picked = (claims[:1] + [c for c in claims[1:] if re.search(r"\d", c["text"])] + claims[1:])
    unique = list({c["text"]: c for c in picked}.values())[:3]
    for number, claim in enumerate(unique, 1):
        source = r["sources"].get(claim["source_ids"][0], {}) if claim["source_ids"] else {}
        REVIEW_ITEMS.append({"id": f"{r['mode']}-{number}", "mode": r["mode"]})
        print(f"[{r['mode']}-{number}] {claim['text']}\n  출처: {source.get('url', '없음')}\n  원문: {excerpt(source.get('text', ''), claim['text']) or '없음'}\n")
print("고른 주장:", len(REVIEW_ITEMS), "| 모델 호출 0회, EXA 호출 0회")

In [ ]:
JUDGMENTS = ("지지됨", "일부", "지지 안 됨")
HUMAN_REVIEW = {item["id"]: "미판정" for item in REVIEW_ITEMS}
# 원문을 읽은 뒤 이렇게 적습니다: HUMAN_REVIEW["parallel-1"] = "지지됨"

rows = []
for mode in ["parallel", "serial", "chained"]:
    ids = [item["id"] for item in REVIEW_ITEMS if item["mode"] == mode]
    counts = Counter(HUMAN_REVIEW[i] for i in ids)
    judged = sum(counts[j] for j in JUDGMENTS)
    rows.append({"구조": mode, **{j: counts[j] for j in JUDGMENTS}, "미판정": counts["미판정"],
                 "판정한 것 중 지지됨 비율": shown(counts["지지됨"] / judged if judged else None, 2)})
show_table(rows)
waiting = sum(v == "미판정" for v in HUMAN_REVIEW.values())
print(f"미판정 {waiting}건은 품질 결과가 아닙니다. 지지됨에도 지지 안 됨에도 세지 않습니다.")
print("모델 호출 0회, EXA 호출 0회")

**관찰할 것.** `문제 수` 표에서 먼저 `출처를 지운 사본` 행을 봅니다. 이 행의 `출처 없는 주장`이 0보다 크면 검사가 실제로 실패할 줄 아는 것입니다. 그다음 세 보고서를 봅니다. 모두 0이라면 형식 위조가 없었다는 뜻이지, 내용이 맞다는 뜻이 아닙니다. 0이 아닌 행이 있다면 합성 노드가 출처를 빠뜨렸거나 바꾼 것입니다. 어느 구조에서 나왔는지와 상관없이 그 주장을 찾아 읽습니다.

**인용 ID가 유효한 것과 뜻이 지지되는 것은 다릅니다.** `validate_report`는 `sections` 안의 주장만 봅니다. 보고서의 `summary` 문장에는 인용 칸이 없어서 검사 대상도 아닙니다. 합성이 요약에서 새로운 결론을 냈다면 이 검사로는 잡히지 않습니다.

원문 발췌는 잘린 일부입니다. 발췌에서 근거를 찾지 못했다고 곧바로 "지지 안 됨"으로 적지 말고, 출력된 URL로 들어가 원문을 읽습니다. 웹 문서의 본문은 `MAX_CHARS`에서 잘려 있으니, Worker가 **그 앞부분만** 보고 주장을 썼다는 점도 함께 생각합니다. 발췌 시작 위치는 주장 속 숫자를 찾아 정한 것이라 같은 숫자가 다른 맥락에 있을 수도 있습니다.

**미판정은 품질 결과가 아닙니다.** 판정표의 `미판정` 칸은 사람이 아직 보지 않았다는 기록입니다. 판정이 하나도 없으면 비율 칸은 `확인 안 됨`입니다. 이 상태에서 "세 구조의 품질이 같다"고 쓰면 틀린 문장입니다. 수업 시간 안에 고른 주장을 다 읽기 어렵다면 구조마다 하나씩이라도 판정하고, 판정 수가 적다고 함께 적습니다.

In [ ]:
optional_parallel = None
if RUN_OPTIONAL:
    announce("같은 구조 안의 변동을 보려고 병렬을 한 번 더", model_calls=16, exa_calls=6)
    optional_parallel = run_research("parallel", PLAN, model=model, **RUN_ARGS)
    show_table(compare_rows([parallel, optional_parallel]))
else:
    print("실행하지 않았습니다. CONFIG 셀의 RUN_OPTIONAL을 True로 바꾸면 모델 호출 약 10~16회, EXA 호출 약 3~6회가 더 듭니다.")

In [ ]:
if optional_parallel is not None:
    show_table([{"비교": "parallel 1회차 대 2회차 (같은 구조)", **url_overlap(parallel, optional_parallel)},
                {"비교": "parallel 대 serial (다른 구조)", **url_overlap(parallel, serial)}])
else:
    print("실행하지 않았습니다. 같은 구조 안의 변동을 재지 않았으므로 구조 간 차이의 크기는 1회 관찰로만 읽습니다.")

**관찰할 것.** 위 선택 셀을 켜고 돌렸다면 표의 두 `Jaccard`를 비교합니다. 같은 구조로 두 번 돌린 값이 다른 구조끼리의 값과 비슷하다면, 구조 사이의 URL 차이는 구조 때문이라고 말할 근거가 약합니다. 같은 구조의 값이 분명히 더 높다면 구조 효과의 후보로 적을 수 있지만, 그래도 2회 관찰입니다. 켜지 않았다면 "실행하지 않았습니다"가 그대로 기록입니다. 재지 않은 변동을 0으로 읽지 않습니다.

## 7. 구조 선택과 전이

**문제.** 오늘의 결과물은 Agent 조직도가 아니라 **구조를 고른 이유**입니다. 비교표가 있어도 그것만으로 구조가 정해지지는 않습니다. 무엇을 포기할지는 업무가 정합니다.

구조를 고를 때 답해야 하는 설계 질문은 여섯 가지입니다.

1. **업무 목표.** 어떤 상태를 완료로 볼까요?
2. **Agent·SubAgent의 필요성.** 코드나 Skill로 충분하지 않은 이유는 무엇인가요?
3. **작업 Graph.** 순차, 병렬, 분기, 루프는 어디에 있나요?
4. **Context·권한.** 각 담당자는 무엇을 보고 무엇을 실행할 수 있나요?
5. **예산·종료.** 언제 재시도하고, 언제 전환하고, 언제 멈추나요?
6. **평가 근거.** 어떤 결과로 선택을 정당화할까요?

아래 셀은 앞 노트북에서 쓴 설계 기록을 읽습니다. 앞 노트북을 돌리지 않았어도 이 셀은 멈추지 않습니다. 비어 있는 항목이 보이면 비어 있다고 표시합니다.

In [ ]:
log = read_design_log()
TOPICS = {"업무 목표": ("업무 목표", "완료"), "Agent·SubAgent의 필요성": ("SubAgent", "Skill", "필요"), "작업 Graph": ("Graph", "흐름"),
          "Context·권한": ("Context", "권한", "무엇을 보고"), "예산·종료": ("예산", "종료", "재시도"), "평가 근거": ("평가", "정당화")}
show_table([{"설계 질문": name, "기록": " / ".join(str(v)[:60] for k, v in log.items() if any(word in k for word in words)) or "비어 있음"}
            for name, words in TOPICS.items()])
print("저장된 기록 전체:", len(log), "건")
for key, value in log.items():
    print(f"- {key}: {str(value)[:120]}".replace("\n", " "))
print("모델 호출 0회, EXA 호출 0회")

In [ ]:
DECISION = {
    "구조": "",        # parallel, serial, chained 또는 내가 바꾼 구조
    "근거 칸": "",     # 위 표의 칸 이름과 그 값을 그대로 적습니다
    "포기한 것": "",   # 이 구조를 골라서 잃는 것
}
show_table([{"칸": k, "내 답": v or "비어 있음"} for k, v in DECISION.items()])
print("모델 호출 0회, EXA 호출 0회")

**전이 연습: 투자 심사.** (`Day03/Day03_01.png`의 판서에서 가져온 업무입니다.) 전혀 다른 업무로 옮겨 봅니다. 어느 회사의 투자 심사는 이렇게 흘러갑니다.

수요 기업이 심사를 요청합니다. 담당자는 내부 자료와 외부 자료를 모으고 기업 인터뷰를 진행합니다. 그 내용으로 양식이 통일된 보고서를 씁니다. 보고서는 50개 절쯤 되고, 재무 수치가 여러 절에 되풀이됩니다. 완성된 보고서는 투자심의위원회로 가서 Go 또는 Stop 판단을 받습니다. Go라면 뒤이어 회계와 감사가 따라옵니다.

오늘 배운 것으로 세 가지를 답해 봅니다.

- **숫자는 어떻게 다룰까요?** 절마다 Worker가 재무 수치를 옮겨 쓰게 하면 어떤 일이 일어날까요? 같은 숫자를 한 곳에서 계산해 참조하게 하려면 무엇이 코드이고 무엇이 모델이어야 할까요?
- **50개 절의 Context는 어떻게 나눌까요?** 앞 절의 내용을 뒤 절에 모두 넘기면 오늘 `연쇄 입력 바이트`에서 본 일이 50번 쌓입니다. 무엇을 요약으로 넘기고, 무엇을 원문 참조로 남기면 안전할까요? 요약에서 빠지면 안 되는 것은 무엇일까요?
- **사람의 문체와 관점은 어디에 담을까요?** 심사역의 판단 기준과 문체는 프롬프트, Skill, 양식, 예시 중 어디에 있어야 할까요? Go/Stop은 모델에게 맡겨도 될까요? 맡기지 않는다면 그래프의 어디에서 멈춰야 할까요?

아래 셀은 이 업무를 작업 표로 쓰고, 의존을 지키며 동시에 시작할 수 있는 묶음을 계산합니다. 03에서 만든 것과 같은 계산입니다. 표의 `담당`은 초안입니다. 고쳐 보세요.

In [ ]:
TASKS = {
    "요청 접수": {"앞 작업": [], "담당": "코드"},
    "내부 자료 수집": {"앞 작업": ["요청 접수"], "담당": "코드"},
    "외부 자료 조사": {"앞 작업": ["요청 접수"], "담당": "모델"},
    "기업 인터뷰": {"앞 작업": ["요청 접수"], "담당": "사람"},
    "재무 수치 계산": {"앞 작업": ["내부 자료 수집"], "담당": "코드"},
    "절별 초안 50개": {"앞 작업": ["내부 자료 수집", "외부 자료 조사", "기업 인터뷰", "재무 수치 계산"], "담당": "모델"},
    "양식과 숫자 검증": {"앞 작업": ["절별 초안 50개"], "담당": "코드"},
    "심사역 검토": {"앞 작업": ["양식과 숫자 검증"], "담당": "사람"},
    "투자심의위원회 Go/Stop": {"앞 작업": ["심사역 검토"], "담당": "사람"},
    "회계·감사": {"앞 작업": ["투자심의위원회 Go/Stop"], "담당": "사람"},
}


def batches(tasks):
    done, order = set(), []
    while len(done) < len(tasks):
        ready = [name for name, task in tasks.items() if name not in done and set(task["앞 작업"]) <= done]
        if not ready:
            raise ValueError("작업 표에 순환 의존이 있습니다.")
        order.append(ready)
        done |= set(ready)
    return order


show_table([{"묶음": i + 1, "동시에 시작 가능": ", ".join(group), "담당": ", ".join(sorted({TASKS[n]["담당"] for n in group}))}
            for i, group in enumerate(batches(TASKS))])
print("담당 집계:", dict(Counter(task["담당"] for task in TASKS.values())), "| 모델 호출 0회, EXA 호출 0회")

**관찰할 것.** 묶음 표에서 **어디에 판단을 맡겼고 어디를 코드로 고정했는지** 읽습니다.

- 여러 작업이 함께 있는 묶음은 병렬 후보입니다. 그 안에 `사람`이 섞여 있으면 묶음이 끝나는 시점은 사람이 정합니다. 모델 쪽을 아무리 빨리 돌려도 그 묶음은 줄어들지 않습니다.
- 묶음이 하나짜리로 이어지는 뒷부분은 순차입니다. 어떤 구조로 만들어도 동시에 돌릴 수 없습니다.
- `담당`이 `모델`인 작업마다 실패했을 때의 다음 행동을 말할 수 있어야 합니다. 말할 수 없다면 04의 실패 분류로 돌아갑니다.
- 재무 수치를 `코드`로 두었다면 모델은 숫자를 계산하지 않고 참조합니다. `모델`로 바꾼다면 이 노트북의 `validate_report` 같은 형식 검사로는 숫자가 맞는지 잡을 수 없습니다.
- 이 표는 그래프가 아닙니다. Go/Stop 뒤의 분기와 검증 실패 뒤의 루프는 묶음 계산에 나오지 않습니다. 작업 표는 **순차와 병렬만** 보여 주므로, 분기와 루프는 직접 덧붙여야 합니다.

## 직접 해보기

네 가지 모두 앞의 셀에서 만든 `PLAN`, `RUN_ARGS`, `model`을 그대로 씁니다. 유료 셀은 첫 줄의 `RUN_EXERCISE`를 `True`로 바꿔야 실행됩니다. 고칠 값은 그 아래 상수로 모아 두었습니다. 실행하기 전에 **무엇이 달라질지 먼저 적어** 두세요. 조건을 하나만 바꿔야 차이를 그 조건 때문이라고 말할 수 있습니다.

### 해보기 1

**자기 주제로 바꿔 실행합니다.** 업무에서 실제로 궁금한 공개 질문을 소주제 셋으로 나눕니다. 고객 이름과 내부 번호는 넣지 않습니다. 주제가 바뀌면 믿을 만한 출처도 바뀌므로, 실행하기 전에 그 주제의 공식 문서 도메인을 골라 `MY_DOMAINS`에 적습니다. 검색 결과가 적다는 이유만으로 목록을 비우거나 넓히지 않습니다. 넓혀야 한다면 왜 그 도메인을 믿을 수 있는지 코드 주석에 적습니다. 모델 호출 약 10~16회, EXA 호출 약 3~6회입니다.

In [ ]:
RUN_EXERCISE = False
MY_TOPIC = "사내 지식 검색 도구를 도입할 때 무엇을 먼저 평가할 것인가"
MY_SUBTOPICS = {"a": "검색 품질은 어떤 지표로 평가하는가", "b": "권한이 다른 문서를 어떻게 분리하는가", "c": "운영 비용은 어디서 가장 많이 드는가"}
MY_DOMAINS = ALLOWED_DOMAINS

if RUN_EXERCISE:
    my_plan = ResearchPlan(topic=MY_TOPIC, subtopics=[SubTopic(id=k, question=q) for k, q in MY_SUBTOPICS.items()])
    announce("해보기 1: 내 주제로 병렬 실행", model_calls=16, exa_calls=6)
    mine = run_research("parallel", my_plan, model=model, **{**RUN_ARGS, "domains": MY_DOMAINS})
    show_table(compare_rows([mine]))
    print(mine["report"]["summary"] if mine["report"] else mine["errors"])
    print(validate_report(mine["report"], mine["sources"]) or "자동 검증 문제 없음")
else:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")

### 해보기 2

**소주제에 일부러 의존 관계를 만듭니다.** 아래 기본값에서 둘째와 셋째 소주제는 첫째의 답을 알아야 제대로 물을 수 있습니다. 병렬에서는 Worker가 앞 답을 모르니 "앞에서 고른 패턴"을 무엇으로 읽는지 봅니다. 연쇄와 비교해 병렬이 무엇을 놓치는지, 그리고 연쇄의 첫 Worker가 고른 답에 뒤 Worker가 얼마나 묶이는지 함께 적습니다. 모델 호출 약 20~32회, EXA 호출 약 6~12회입니다.

In [ ]:
RUN_EXERCISE = False
DEPENDENT_SUBTOPICS = {"d1": "리서치 업무에 가장 널리 권장되는 멀티에이전트 패턴은 무엇인가",
                       "d2": "앞 소주제에서 고른 패턴이 실패하는 대표적인 경우는 무엇인가",
                       "d3": "앞 소주제의 실패를 줄이기 위해 공개 문서가 권하는 장치는 무엇인가"}

if RUN_EXERCISE:
    dependent = ResearchPlan(topic="리서치 업무의 멀티에이전트 패턴과 그 실패", subtopics=[SubTopic(id=k, question=q) for k, q in DEPENDENT_SUBTOPICS.items()])
    announce("해보기 2: 의존이 있는 소주제로 병렬과 연쇄 비교", model_calls=32, exa_calls=12)
    dep_runs = [run_research(mode, dependent, model=model, **RUN_ARGS) for mode in ["parallel", "chained"]]
    show_table(compare_rows(dep_runs))
    show_table([{"비교": "parallel 대 chained", **url_overlap(*dep_runs)}])
    for run in dep_runs:
        print(run["mode"], "검색어:", run["queries"])
else:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")

### 해보기 3

**본문 길이를 절반으로 줄입니다.** 바꾸는 것은 `MAX_CHARS` 하나입니다. `입력 토큰`이 얼마나 줄었는지, `자동 검증`의 문제 수가 달라졌는지, `uncertainty`가 늘었는지를 봅니다. 토큰이 줄어도 조건과 예외가 잘려 나갔다면 그 손실을 함께 적습니다. 비용이 줄어든 것만 보고 개선이라고 쓰지 않습니다. 모델 호출 약 10~16회, EXA 호출 약 3~6회입니다.

In [ ]:
RUN_EXERCISE = False
SHORT_CHARS = MAX_CHARS // 2

if RUN_EXERCISE:
    announce("해보기 3: 본문 길이 절반으로 병렬 실행", model_calls=16, exa_calls=6)
    short = run_research("parallel", PLAN, model=model, **{**RUN_ARGS, "max_chars": SHORT_CHARS})
    show_table(compare_rows([parallel, short]))
    show_table([{"비교": "기본 대 절반", **url_overlap(parallel, short)}])
    print("불확실성 수:", {r["mode"] + str(i): sum(len(f["uncertainty"]) for f in r["findings"].values()) for i, r in enumerate([parallel, short])})
    print(validate_report(short["report"], short["sources"]) or "자동 검증 문제 없음")
else:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")

### 해보기 4

**합성 뒤에 검증 루프를 더합니다.** 아래 골격은 모델 없이 도는 뼈대입니다. `draft`는 첫 번째 시도에서 일부러 출처를 지운 보고서를 내고, 다시 부르면 원래 보고서를 냅니다. `flows_of`의 `순환`과 `조건부 간선`을 읽고, 이 그래프를 순차, 병렬, 분기, 루프 중 어떤 흐름들의 조합으로 설명할 수 있는지 적어 보세요. 그다음 `draft`를 `model.with_structured_output(ResearchReport, method="json_schema")`를 부르는 노드로 바꾸면 실제 재작성 루프가 됩니다. 그때는 `MAX_REWRITES`가 모델 호출을 얼마나 늘리는지부터 계산하고 `announce`를 더합니다.

In [ ]:
MAX_REWRITES = 2


class LoopState(TypedDict, total=False):
    report: dict
    problems: list
    attempts: int


def draft(state):
    report = json.loads(json.dumps(parallel["report"] or {"summary": "", "sections": []}))
    attempts = state.get("attempts", 0) + 1
    if attempts == 1:
        for section in report["sections"]:
            for claim in section["claims"]:
                claim["source_ids"] = []
    return {"report": report, "attempts": attempts}


def check(state):
    return {"problems": validate_report(state["report"], parallel["sources"])}


loop = StateGraph(LoopState)
loop.add_node("draft", draft)
loop.add_node("check", check)
loop.add_edge(START, "draft")
loop.add_edge("draft", "check")
loop.add_conditional_edges("check", lambda s: "draft" if s["problems"] and s["attempts"] <= MAX_REWRITES else END, ["draft", END])
loop_app = loop.compile()
show_graph(loop_app)
print("flows_of:", flows_of(loop_app))
ended = loop_app.invoke({})
print("시도 횟수:", ended["attempts"], "| 마지막 문제 수:", len(ended["problems"]), "| 모델 호출 0회, EXA 호출 0회")

In [ ]:
answer = " / ".join(f"{k}: {v}" for k, v in DECISION.items() if v)
if answer:
    print("기록했습니다:", write_design_log("리서치 구조 선택", answer).name)
else:
    print("DECISION이 비어 있어 기록하지 않았습니다. 7절의 칸을 채운 뒤 이 셀을 다시 실행하세요.")

## 정리

오늘 전체를 한 질문으로 다시 쓰면 **"어디에 판단을 맡길까?"**입니다.

- **코드에 고정한 것.** 외부로 나가는 질의의 검사, 허용 도메인, 검색 한도, `source_id`의 등록, 결과 합치기, 인용 형식 검증. 이것들은 모델이 잘하고 못하고와 상관없이 지켜져야 하는 것들입니다.
- **모델에 맡긴 것.** 검색어, 검색을 나눌지 묶을지, 무엇을 주장으로 쓸지, 무엇이 불확실한지, 어떻게 합칠지. 이 판단은 돌릴 때마다 달라질 수 있으므로 측정과 검증이 함께 있어야 합니다.
- **사람에게 남긴 것.** 그 문서가 그 주장을 정말 뒷받침하는지, 어떤 구조로 갈지, 무엇을 포기할지.

**구조를 고르는 일도 같은 질문입니다.** `parallel`과 `serial`이 바꾼 것은 동시 실행이고, `serial`과 `chained`가 바꾼 것은 앞 결과의 전달입니다. 소주제가 독립적이면 정보 흐름의 값을 치를 이유가 줄어들고, 서로 기대면 시간을 내주더라도 앞 결과를 넘기는 쪽이 후보가 됩니다. 어느 쪽이 맞는지는 패턴 이름이 아니라 내 업무의 의존 관계와 오늘 표의 칸이 알려 줍니다. 이번 비교는 각 구조를 한 번씩 돌린 관찰이므로, 구조의 우열을 판정한 것이 아니라 **어디를 재야 하는지**를 익힌 것입니다.

오늘 처음 쓴 고객 요청으로 돌아가 보세요. 결제 조회와 규정 검색을 나눈 이유, 환불 실행을 게이트 뒤에 둔 이유, 평가 근거를 먼저 정한 이유가 모두 이 질문의 답이었습니다. 이 노트북의 전이 연습에 적은 투자 심사 표로 같은 설명을 할 수 있다면 오늘 과정의 목표를 이룬 것입니다.

## 확인 질문

자기 말로 답해 보세요. 표의 칸 이름을 근거로 들 수 있어야 합니다.

1. 직렬과 연쇄를 나눠 본 이유는 무엇인가요? 둘의 차이는 시간과 정보 흐름 중 어디에 나타나나요?
2. URL 겹침이 낮은 두 실행의 품질 차이를 구조 탓으로 말하면 왜 틀린가요?
3. `assert_public_query`가 막는 것과 막지 못하는 것은 무엇인가요? 2절의 통과한 질의를 예로 들어 설명해 보세요.
4. 인용 검증을 통과한 주장을 사람이 왜 다시 보나요? 판정표의 미판정을 어떻게 보고해야 하나요?
5. 투자 심사에 고른 구조를 설계 질문 여섯 가지로 설명해 보세요.
6. `EXA 비용(USD)`이나 `입력 토큰` 칸이 `확인 안 됨`일 때 합계를 어떻게 보고해야 하나요? 왜 0으로 더하면 안 되나요?